# Look at different beam hardening parameters

Reviewer2 asked us why we didn't correct for beam hardening, see https://github.com/habi/sticklebacks-manuscript/issues/66
Instead of just telling them "we didn't need to", let's answer them with some images.
This notebook started from a copy of the PreviewScans notebook.

In [ ]:
# Load the modules we need
import platform
import os
import glob
import pandas
import imageio
import numpy
import skimage
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib_scalebar.scalebar import ScaleBar
import seaborn
import dask
import dask_image.imread
from dask.distributed import Client, LocalCluster
from tqdm.auto import tqdm, trange

In [ ]:
# Load our own log file parsing code
# This is loaded as a submodule to alleviate excessive copy-pasting between *all* projects we do
# See https://github.com/habi/BrukerSkyScanLogfileRuminator for details on its inner workings
import BrukerSkyScanLogfileRuminator.parsing_functions as logparse

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
# We use the fast internal SSD for speed reasons
import tempfile

if 'Linux' in platform.system():
    # Check if me mounted the FastSSD, otherwise go to standard tmp file
    if os.path.exists(os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')):
        tmp = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD', 'tmp')
    else:
        tmp = tempfile.gettempdir()
elif 'Darwin' in platform.system():
    tmp = tempfile.gettempdir()
else:
    if 'anaklin' in platform.node():
        tmp = os.path.join('F:\\tmp')
    else:
        tmp = os.path.join('D:\\tmp')
dask.config.set({'temporary_directory': tmp})
print('Dask temporary files go to %s' % dask.config.get('temporary_directory'))

In [ ]:
from dask.distributed import Client

client = Client()

In [ ]:
client

In [ ]:
seaborn.set_context('paper')
seaborn.set_theme(style='whitegrid')

In [ ]:
# Set up figure defaults
plt.rc(
    'image', cmap='gray', interpolation='nearest'
)  # Display all images in b&w and with 'nearest' interpolation
# plt.rcParams['figure.figsize'] = (16 * 0.618, 9 * 0.618)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 300

In [ ]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

In [ ]:
# Display all plots identically
lines = 3
# And then do something like
# plt.subplot(lines, int(numpy.ceil(len(Data) / float(lines))), c + 1)

Since the (tomographic) data can reside on different drives we set a folder to use below

In [ ]:
# Different locations if running either on Linux or Windows
FastSSD = True
if 'Linux' in platform.system():
    if FastSSD:
        BasePath = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
    else:
        BasePath = os.path.join(
            os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback'
        )
elif 'Windows' in platform.system():
    if FastSSD:
        BasePath = os.path.join('F:\\')
    else:
        BasePath = os.path.join('N:\\')
if 'research_storage_ben' in BasePath:
    Root = os.path.join(BasePath)
else:
    Root = os.path.join(BasePath, 'IEE Stickleback')
# Force reading from Bens research storage folder
# Root = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
print('We are loading all the data from %s' % Root)

Now that we are set up, actually start to load/ingest the data.

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files present on disk
# Using os.walk is way faster than using recursive glob.glob
# Not sorting the found logfiles is also making it quicker
Data['LogFile'] = [
    os.path.join(root, name)
    for root, dirs, files in os.walk(Root)
    for name in files
    if name.endswith((".log"))
]

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]

In [ ]:
# Generate us a shorter folder name for easier printing
Data['FolderShort'] = [f[len(Root) + 1 :] for f in Data['Folder']]

In [ ]:
Data.sample(n=5)

In [ ]:
# We reconstructed the bottom part of BucketH with beam hardening correction `range(0,110,10)`.
# So for this notebook we can drop all the other folders.
Data = Data[Data['FolderShort'].str.contains('BucketOfFish_H', na=False)].reset_index(
    drop=True
)

In [ ]:
# Show a (small) sampler of the loaded data as a first check
Data.sample(n=5)

In [ ]:
# Check for samples which are not yet reconstructed
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if (
        'proj' in row.Folder
        and '_14/proj2' not in row.Folder
        and '_15/proj2' not in row.Folder
    ):
        # Sticklebucket_14/proj2/Sticklebucket_14~00.log and
        # Sticklebucket_15/proj2/Sticklebucket_15~00.log are failed scans where we cannot do a reconstruction, so we exclude them
        # If there's nothing with 'rec*' on the same level, then tell us
        if not glob.glob(row.Folder.replace('proj', '*rec*')):
            print(
                '- %s is missing matching reconstructions'
                % row.LogFile[len(Root) + 1 :]
            )

In [ ]:
# Search for any .csv files in each folder.
# These are only generated when the "X/Y Alignment With a Reference Scan" was performed in NRecon.
# If those files do *not* exist we have missed to do it and should correct for this.
Data['XYAlignment'] = [glob.glob(os.path.join(f, '*T*.csv')) for f in Data['Folder']]

In [ ]:
# Display samples which are missing the .csv-files for the XY-alignment
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row['Folder']:
        if not row['XYAlignment']:
            if not any(
                x in row.LogFile
                for x in [
                    'rectmp.log',  # because we only exclude temporary logfiles in a later step
                    '14/proj',  # *All* scans of Sticklebucket 14 are missing the alignment files and cannot be aligned.
                    '15/proj',  # *All* scans of Sticklebucket 15 are missing the alignment files and cannot be aligned
                ]
            ):
                print('- %s has *not* been X/Y aligned' % row.LogFile[len(Root) + 1 :])

In [ ]:
# Get rid of all the logfiles from all the folders that might be on disk but that we don't want to load the data from
for c, row in Data.iterrows():
    if os.path.split(row.Folder)[-1] == 'proj':  # drop all projections folders
        Data.drop([c], inplace=True)
    elif (
        'ucket' not in row.Folder
    ):  # Remove all test scans which are not named 'Sticklbucket_*' or something else containing 'ucket'
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Exclude all log files that we write in ourselves
        Data.drop([c], inplace=True)
    elif os.path.split(row.LogFile)[1].startswith(
        '._'
    ):  # Remove macos metadata files for files on external storage
        Data.drop([c], inplace=True)
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

In [ ]:
# Generate us some meaningful colums in the dataframe
Data['Sample'] = [
    os.path.basename(log).replace('_rec.log', '')
    for log in tqdm(Data['LogFile'], leave=False)
]
Data['Scan'] = [
    os.path.basename(os.path.dirname(log)) for log in tqdm(Data['LogFile'], leave=False)
]

In [ ]:
# Show the data from the last loaded scans
Data.tail(n=5)

In [ ]:
# Drop the single 'rec' folder for this notebook
Data = Data[Data['Scan'] != 'rec'].reset_index(drop=True)

In [ ]:
# Load the file names of all the reconstructions of all the scans
Data['Filenames Reconstructions'] = [
    sorted(glob.glob(os.path.join(f, '*rec0*.png'))) for f in Data['Folder']
]
# How many reconstructions do we have?
Data['NumberOfReconstructions'] = [len(r) for r in Data['Filenames Reconstructions']]

In [ ]:
# Drop samples which have either not been reconstructed yet or of which we deleted the reconstructions with
Data = Data[Data['NumberOfReconstructions'] > 0]
# Reset the dataframe count/index for easier indexing afterwards
Data.reset_index(drop=True, inplace=True)
print('We have %s folders with reconstructions' % (len(Data)))

In [ ]:
# Get parameters to doublecheck from logfiles
Data['Voxelsize'] = [
    logparse.pixelsize(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['Filter'] = [
    logparse.whichfilter(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['Exposuretime'] = [
    logparse.exposuretime(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['Scanner'] = [logparse.scanner(log) for log in tqdm(Data['LogFile'], leave=False)]
Data['Averaging'] = [
    logparse.averaging(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['ProjectionSize'] = [
    logparse.projection_size(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['RotationStep'] = [
    logparse.rotationstep(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['Grayvalue'] = [
    logparse.reconstruction_grayvalue(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['RingartefactCorrection'] = [
    logparse.ringremoval(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['BeamHardeningCorrection'] = [
    logparse.beamhardening(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['DefectPixelMasking'] = [
    logparse.defectpixelmasking(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['Scan date'] = [
    logparse.scandate(log) for log in tqdm(Data['LogFile'], leave=False)
]
Data['Slices_Start'] = [
    logparse.slice_first(logfile) for logfile in tqdm(Data['LogFile'], leave=False)
]
Data['Slices_End'] = [
    logparse.slice_last(logfile) for logfile in tqdm(Data['LogFile'], leave=False)
]
Data['NumRec'] = [
    logparse.slice_number(logfile) for logfile in tqdm(Data['LogFile'], leave=False)
]

In [ ]:
# Usually having NAN for no beam hardening is good, but here we specifically want 0 as value.
Data['BeamHardeningCorrection'] = Data['BeamHardeningCorrection'].fillna(0)

In [ ]:
# Sort dataframe based on beam hardening correction
Data.sort_values(by=['BeamHardeningCorrection'], ignore_index=True, inplace=True)

In [ ]:
Data

Now we 'load' all reconstructions from disks into stacks.

In [ ]:
# Load all reconstructions into ephemereal DASK arrays, with a nice progress bar...
import warnings

warnings.filterwarnings(  # Silence skimage/pims warning about plugin parameter
    "ignore",
    category=FutureWarning,
    message=".*The plugin infrastructure in `skimage.io`.*",
)
Reconstructions = [None] * len(Data)
for c, row in tqdm(Data.iterrows(), desc='Loading reconstructions', total=len(Data)):
    Reconstructions[c] = dask_image.imread.imread(
        os.path.join(row['Folder'], '*rec*.png')
    )[
        :, :, :, 0
    ]  # Get rid of the color channel immediately

In [ ]:
# Nice (dask) display of a random reconstruction stack
Reconstructions[numpy.random.randint(len(Reconstructions))]

In [ ]:
# The three cardinal directions
directions = ['Axial', 'Frontal', 'Median']

In [ ]:
# Read or calculate the directional MIPs, put them into the dataframe and save them to disk
for d, direction in enumerate(directions):
    Data['MIP_' + direction] = ''
for c, row in tqdm(Data.iterrows(), desc='Working on MIPs', total=len(Data)):
    for d, direction in tqdm(
        enumerate(directions),
        desc='%s/%s' % (row['Sample'], row['Scan']),
        leave=False,
        total=len(directions),
    ):
        outfilepath = os.path.join(
            os.path.dirname(row['Folder']),
            '%s.%s.MIP.%s.png' % (row['Sample'], row['Scan'], direction),
        )
        if not os.path.exists(outfilepath):
            # Generate and save MIP
            imageio.imwrite(
                outfilepath, Reconstructions[c].max(axis=d).compute().astype('uint8')
            )
        Data.at[c, 'MIP_' + direction] = dask_image.imread.imread(outfilepath).squeeze()

In [ ]:
# Show/save MIP slices together
for row in tqdm(
    Data.itertuples(), desc='Saving overview of MIP images', total=len(Data)
):
    outfilepath = os.path.join(
        os.path.dirname(row.Folder), '%s.%s.MIPs.png' % (row.Sample, row.Scan)
    )
    if not os.path.exists(outfilepath):
        for d, direction in tqdm(
            enumerate(directions),
            desc='%s/%s' % (row.Sample, row.Scan),
            leave=False,
            total=len(directions),
        ):
            plt.subplot(1, 3, d + 1)
            plt.imshow(getattr(row, f'MIP_{direction}'))
            plt.gca().add_artist(ScaleBar(row.Voxelsize, 'um'))
            plt.title('%s MIP' % direction)
            plt.axis('off')
            plt.title('%s MIP' % (direction))
        plt.suptitle(
            f"MIPs of reconstructions with beam hardening correction of {row.BeamHardeningCorrection}%"
        )
        plt.tight_layout()
        plt.savefig(outfilepath, transparent=True, bbox_inches='tight')
        plt.show()

For further checking the data, we look at the gray values and gray value histograms of the reconstructions.
This helps us to find scans that have not been reconstructed well and might either need to be repeated or simply re-reconstructed.

In [ ]:
def overeexposecheck(
    item, threshold=222, howmanypercent=0.02, whichone='Axial', verbose=False
):
    '''Function to check if a certain amount of voxels are brighter than a certain value'''
    if (Data['MIP_%s' % whichone][item] > threshold).sum() > (
        Data['MIP_%s' % whichone][item].size * howmanypercent / 100
    ):
        if verbose:
            plt.imshow(Data['MIP_%s' % whichone][item])
            plt.imshow(
                dask.array.ma.masked_less(
                    Data['MIP_%s' % whichone][item] > threshold, 1
                ).compute(),
                cmap='viridis_r',
                alpha=0.5,
            )
            plt.title(
                '%s/%s\n%s px of %s Mpixels (>%s%%) are brighter '
                'than %s'
                % (
                    Data['Sample'][item],
                    Data['Scan'][item],
                    (Data['MIP_%s' % whichone][item] > threshold).sum().compute(),
                    round(1e-6 * Data['MIP_%s' % whichone][item].size, 2),
                    howmanypercent,
                    threshold,
                )
            )
            plt.axis('off')
            plt.gca().add_artist(ScaleBar(Data['Voxelsize'][item], 'um'))
            plt.show()
        return True
    else:
        return False

In [ ]:
# Check if 'too much' of the MIP is overexposed
Data['OverExposed'] = [
    overeexposecheck(c, howmanypercent=0.05, whichone='Frontal', verbose=True)
    for c, row in Data.iterrows()
]

In [ ]:
# Prepare the histograms of all the images
# Caveat: dask.da.histogram returns histogram AND bins, making each histogram a 'nested' list of [h, b]
# Computation happens on demand later on
Data['Histogram'] = [
    dask.array.histogram(rec, bins=2**8, range=[0, 2**8]) for rec in Reconstructions
]

In [ ]:
# Make us an adapted colormap
cmap = plt.get_cmap('viridis')
norm = matplotlib.colors.Normalize(vmin=0, vmax=100)

In [ ]:
# # Plot single histograms
# for row in tqdm(Data.itertuples(), total=len(Data), leave=False):
#     outfilepath = os.path.join(os.path.dirname(row.Folder),
#                                 '%s.%s.Histogram.png' % (row.Sample,
#                                                          row.Scan))
#     if not os.path.exists(outfilepath):
#         plt.semilogy(row.Histogram[0].compute(),
#                      color=cmap(norm(row.BeamHardeningCorrection)),
#                      label=f'{row.BeamHardeningCorrection}%')
#         plt.xlim([0, 2**8])
#         plt.ylim(ymin=10**0)
#         plt.title(f"{row.Sample}/{row.Scan}\n{row.NumberOfReconstructions} reconstructions with beam hardening correction of {row.BeamHardeningCorrection}%")
#         seaborn.despine()
#         plt.savefig(outfilepath,
#                     transparent=True,
#                     bbox_inches='tight')
#         plt.show()

In [ ]:
# Plot histograms together
outfilepath = os.path.join(
    os.path.dirname(Data.iloc[0]['Folder']),
    f"{row.Sample}.Combined.Histograms.Beamhardening.png",
)
if not os.path.exists(outfilepath):
    for row in tqdm(Data.itertuples(), total=len(Data), leave=False):
        plt.semilogy(
            row.Histogram[0].compute(),
            color=cmap(norm(row.BeamHardeningCorrection)),
            label=f'{row.BeamHardeningCorrection}%',
        )
    plt.xlim([0, 2**8])
    plt.ylim(ymin=10**0)
    plt.title('Combined histograms')
    plt.legend()
    seaborn.despine()
    plt.savefig(outfilepath, transparent=True, bbox_inches='tight')
    plt.show()

In [ ]:
def cuppingplotter(whichscan):
    '''
    Answer the beamhardening artefact correction by showing that we don't really have any cupping going on
    Let's show the plot through the central, horizontal wall.'''
    outfilepath = os.path.join(
        os.path.dirname(Data.iloc[whichscan]['Folder']),
        f"{Data.iloc[whichscan]['Sample']}.{Data.iloc[whichscan]['Scan']}.Cupping.png",
    )
    if not os.path.exists(outfilepath):
        # Load central image
        centralimage = Reconstructions[whichscan][
            Reconstructions[whichscan].shape[1] // 2
        ].compute()

        # Profile region setup
        profilelevel = centralimage.shape[1] // 2 + 25
        profilewidth = 2000
        profileheight = 60

        # Calculate starts and stops
        xcentre = centralimage.shape[1] // 2
        xstart = max(0, xcentre - profilewidth // 2)
        xstop = min(centralimage.shape[1], xcentre + profilewidth // 2)
        ystart = max(0, profilelevel - profileheight // 2)
        ystop = min(centralimage.shape[0], profilelevel + profileheight // 2)

        # Horizontal gray-value profile
        gv_profile = centralimage[ystart:ystop, xstart:xstop].mean(axis=0)
        print(
            f"BH {Data.iloc[whichscan]['BeamHardeningCorrection']}: GV profile varying from {round(gv_profile.min())} to {round(gv_profile.max())} ({round(gv_profile.max() - gv_profile.min())})."
        )
        x = numpy.arange(xstart, xstop)
        # Normalize gray value profile to display above the marked region
        # Directly with coordinates, so we can simply plot it later
        gv_profile_normalized = (
            ystart  # start above the rectangle
            - (gv_profile - gv_profile.min())
            / (numpy.ptp(gv_profile) or 1)
            * 0.618
            * profilelevel  # scale to golden ratio of remaining height
        )
        # Gray-value profile relative to its mean
        gv_mean = numpy.mean(gv_profile)
        gv_std = numpy.std(gv_profile)
        gv_cv = 100 * gv_std / gv_mean
        gv_profile_relative = 100 * (gv_profile - gv_mean) / gv_mean

        # Plot everything
        fig = plt.figure(figsize=(14, 8))
        gs = fig.add_gridspec(
            2,
            2,
            width_ratios=[1.1, 1],
            height_ratios=[1, 1],
            left=0.05,
            right=0.97,
            bottom=0.08,
            top=0.95,
            wspace=0.3,
            hspace=0.3,
        )
        ax0 = fig.add_subplot(gs[:, 0])  # Image spans both rows
        ax1 = fig.add_subplot(gs[0, 1])  # Relative gray-value profile
        ax2 = fig.add_subplot(gs[1, 1])  # Mean gray value
        # Image and profile overlay
        ax0.imshow(skimage.exposure.equalize_adapthist(centralimage))
        ax0.add_patch(
            patches.Rectangle(
                (xstart, ystart),
                xstop - xstart,
                profileheight,
                edgecolor=None,
                linewidth=0,
                alpha=0.618,
                label=f"Profile region (approx. wall)\n({round(profilewidth * Data['Voxelsize'][whichscan] / 1000, 1)} x {round(profileheight * Data['Voxelsize'][whichscan] / 1000,1)} mm)",
            )
        )
        ax0.plot(
            x,
            gv_profile_normalized,
            label=f'Normalized gray value\nMean across wall thickness',
        )
        ax0.legend()
        ax0.axis('off')
        ax0.add_artist(ScaleBar(Data['Voxelsize'][whichscan], 'um'))
        # Original gray values
        ax1.plot(x * Data['Voxelsize'][whichscan] / 1000, gv_profile)
        ax1.set_xlabel('Horizontal position [mm]')
        ax1.set_ylabel('Mean gray value')
        ax1.set_ylim([0, 30])
        ax1.set_title('Gray-value profile (not normalized)')
        # Plot relative gray values
        ax2.plot(
            x * Data['Voxelsize'][whichscan] / 1000,
            gv_profile_relative,
        )
        ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax2.set_xlabel('Horizontal position [mm]')
        ax2.set_ylabel('Deviation from mean gray value [%]')
        ax2.set_title(f'Relative gray-value profile (CV = {gv_cv:.2f}%)')
        ax2.set_ylim([-30, 30])
        plt.suptitle(
            f"{Data.iloc[whichscan]['Sample']}, BHC {Data.iloc[whichscan]['BeamHardeningCorrection']}%"
        )
        plt.savefig(outfilepath, transparent=True, bbox_inches='tight')
        plt.show()
    return

In [ ]:
for row in tqdm(Data.itertuples(), total=len(Data), leave=False):
    cuppingplotter(row.Index)

In [ ]:
# Now combine the plots in ImageJ and save them to a .gif